[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/02_reward_models/02_reward_model.ipynb)

# 02 · 奖励模型：Bradley-Terry 与 reward hacking

<span style="background:#e8f5e9;border-radius:4px;padding:2px 8px;font-size:0.85em">🖥️ CPU 全程可跑（纯 PyTorch 小模型 + 合成数据，无下载、约 2 分钟）</span>

本 notebook 在一个**完全可控的合成偏好世界**里复现奖励模型的核心现象：

1. **合成偏好世界**：定义已知的"真奖励" `gold_r(x)`，用 Bradley-Terry 概率模拟（带噪声的）标注者，生成 500 对偏好数据；
2. **训练 RM**：小 MLP reward head + BT loss，看 loss / 准确率曲线与 held-out 偏好准确率；
3. **长度偏好注入**：让标注者略偏爱"长"，量化训出的 RM 的系统性长度偏差；
4. **Overoptimization 实验（核心）**：用 best-of-n 对 RM 优化，复现 [Gao 2022] 的 proxy/gold 分叉曲线，并以 $\mathrm{KL}_{\text{BoN}}(n)=\log n-\frac{n-1}{n}$ 为横轴重画。

合成世界的好处：**gold reward 已知**——这在真实 RLHF 中是永远拿不到的上帝视角，却是理解 reward hacking 最快的路径（[Gao 2022] 的实验设计哲学相同：用大 RM 充当 gold）。

> 配套讲解：`02_讲解.html` · 参考文献：[Christiano 2017] [Stiennon 2020] [Bradley & Terry 1952] [Lambert 2024, RewardBench] [Gao 2022]


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
torch.set_default_dtype(torch.float64)  # 小实验，用 double 让数值更干净

DIM = 8        # "回答"的特征维度
LEN = 7        # 第 7 维约定为"长度"特征（标准化后的对数长度）
print(torch.__version__)

## Part 1 · 合成偏好世界：已知的 gold reward + BT 标注者

把一个"回答"抽象成特征向量 $x \in \mathbb{R}^8$（可以想象为：相关性、正确性、清晰度……以及 $x_7$ = 长度）。**真奖励**由"内容质量"与"啰嗦惩罚"两部分组成：

$$ r_{\text{base}}(x) = 2\tanh(x_0) + x_1 x_2 - 0.3\,x_3^2 + \sin(x_4), \qquad
   r^\*(x) = r_{\text{base}}(x) - 0.7\,\mathrm{relu}(x_7)^2 $$

注意两个刻意的设计：
- $x_5, x_6$ 是**无关特征**（真奖励完全不看）；
- 长度 $x_7$ **不加分**，偏长（$x_7>0$）还会被二次方扣分——"啰嗦不是优点"。

**标注者**看不到 $r^\*$ 的数值，只做成对比较，行为服从 Bradley-Terry 模型：

$$ P(a \succ b) = \sigma\big(\beta\,(r^\*(a) - r^\*(b))\big), \qquad \beta = 2 $$

再叠加 **10% 的随机翻转**（疲劳、误点、口味差异）。每条偏好样本就是 `(chosen, rejected)` 一对特征向量——与真实 RLHF 的 `(prompt, chosen, rejected)` 同构（这里把 prompt 的影响吸收进特征里）。

## Part 2 · 用 BT loss 训练 RM

RM 是一个小 MLP $r_\phi:\mathbb{R}^8\to\mathbb{R}$（对应真实 RM 的 "LLM backbone + 标量 value head"），训练损失就是讲解第 2 节推出的负对数 BT 似然：

$$ \mathcal{L}(\phi) = -\mathbb{E}\big[\log\sigma\big(r_\phi(x_w) - r_\phi(x_l)\big)\big] $$

我们追踪训练 loss 与 held-out 偏好准确率。**准确率天花板不是 100%**：标注本身有 BT 随机性 + 10% 翻转，连"完美 RM"（直接用 $r^\*$ 打分）也只能拿到约 80% —— 我们会把这条天花板一起画出来。

In [ ]:
def base_r(x):
    # 内容质量：特征的非线性函数；x: (..., 8) tensor
    return (2 * torch.tanh(x[..., 0])
            + x[..., 1] * x[..., 2]
            - 0.3 * x[..., 3] ** 2
            + torch.sin(x[..., 4]))

def gold_r(x):
    # 真奖励 = 内容质量 - 啰嗦惩罚
    return base_r(x) - 0.7 * F.relu(x[..., LEN]) ** 2

def simulate_annotator(xa, xb, beta=2.0, flip=0.10, length_bias=0.0, gen=None):
    # BT 概率下采样偏好 + flip 比例的随机翻转。
    # length_bias = 0：理想标注者，按真奖励 gold_r 比较；
    # length_bias > 0：带偏标注者——只看内容质量 + γ·长度，
    #                  既偏爱长回答、又感知不到啰嗦的真实代价（Part 3 用）。
    if gen is None:
        gen = torch.Generator().manual_seed(0)
    if length_bias == 0.0:
        score_a, score_b = gold_r(xa), gold_r(xb)
    else:
        score_a = base_r(xa) + length_bias * xa[..., LEN]
        score_b = base_r(xb) + length_bias * xb[..., LEN]
    p_a_wins = torch.sigmoid(beta * (score_a - score_b))
    a_wins = torch.rand(p_a_wins.shape, generator=gen) < p_a_wins
    flipped = torch.rand(p_a_wins.shape, generator=gen) < flip
    a_wins = a_wins ^ flipped
    chosen   = torch.where(a_wins.unsqueeze(-1), xa, xb)
    rejected = torch.where(a_wins.unsqueeze(-1), xb, xa)
    return chosen, rejected

def make_pref_dataset(n_pairs=500, length_bias=0.0, seed=0):
    gen = torch.Generator().manual_seed(seed)
    xa = torch.randn(n_pairs, DIM, generator=gen)
    xb = torch.randn(n_pairs, DIM, generator=gen)
    return simulate_annotator(xa, xb, length_bias=length_bias, gen=gen)

chosen, rejected = make_pref_dataset(500)
n_train = 400
tr_c, tr_r = chosen[:n_train], rejected[:n_train]
ho_c, ho_r = chosen[n_train:], rejected[n_train:]

# 完美 RM（直接用 gold_r 打分）的 held-out 准确率 = 噪声决定的天花板
ceiling = (gold_r(ho_c) > gold_r(ho_r)).double().mean().item()
print(f"偏好对：train {len(tr_c)} / held-out {len(ho_c)}")
print(f"完美 RM 的 held-out 准确率（天花板）: {ceiling:.3f}")

In [ ]:
def make_rm(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(DIM, 16), nn.Tanh(),
                         nn.Linear(16, 16), nn.Tanh(),
                         nn.Linear(16, 1))

def rm_score(model, x):
    return model(x).squeeze(-1)

def pref_acc(model, c, r):
    with torch.no_grad():
        return (rm_score(model, c) > rm_score(model, r)).double().mean().item()

def train_rm(tr_c, tr_r, ho_c, ho_r, epochs=600, lr=5e-3, weight_decay=1e-3, seed=0):
    model = make_rm(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"loss": [], "acc_tr": [], "acc_ho": []}
    for ep in range(epochs):
        d = rm_score(model, tr_c) - rm_score(model, tr_r)
        loss = -F.logsigmoid(d).mean()          # BT 负对数似然（数值稳定写法）
        opt.zero_grad(); loss.backward(); opt.step()
        hist["loss"].append(loss.item())
        hist["acc_tr"].append(pref_acc(model, tr_c, tr_r))
        hist["acc_ho"].append(pref_acc(model, ho_c, ho_r))
    return model, hist

rm, hist = train_rm(tr_c, tr_r, ho_c, ho_r)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist["loss"]); axes[0].axhline(math.log(2), ls=":", c="gray", label="log 2（随机水平）")
axes[0].set_title("BT loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist["acc_tr"], label="train"); axes[1].plot(hist["acc_ho"], label="held-out")
axes[1].axhline(ceiling, ls=":", c="gray", label=f"天花板 {ceiling:.2f}")
axes[1].set_title("偏好准确率"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"最终 held-out 准确率: {hist['acc_ho'][-1]:.3f}（天花板 {ceiling:.3f}）")

## Part 3 · 长度偏好注入：RM 忠实地学会了标注者的坏习惯

现在换一批**带偏标注者**：他们按 $r_{\text{base}}(x) + \gamma\, x_7$（$\gamma = 2$）做比较——偏爱长回答，并且**感知不到啰嗦的真实代价**（看到长篇大论就觉得"认真"）。注意 **gold reward 没有变**——变的只是标注行为。

用同样的流程训一个新 RM，然后做受控对照：取同一批随机 $x$，只把长度维拨到 $+2$ 与 $-2$，看打分差

$$ \Delta_{\text{len}} = \mathbb{E}_x\big[r(x_{\text{len}=+2}) - r(x_{\text{len}=-2})\big] $$

gold reward 的 $\Delta_{\text{len}}$ 应当为负（偏长被扣分），而带偏 RM 的 $\Delta_{\text{len}}$ 会显著为正。这正是真实 RLHF 中 "RLHF 之后模型越说越长" 的最小复现（对照讲解第 4 节与 RewardBench 的 Chat-Hard 维度）。

In [ ]:
# 带长度偏好的标注者 → 偏好数据 → 训练"带偏 RM"
b_chosen, b_rejected = make_pref_dataset(500, length_bias=2.0, seed=1)
rm_biased, _ = train_rm(b_chosen[:400], b_rejected[:400], b_chosen[400:], b_rejected[400:], seed=1)

def length_gap(score_fn, n=2000, seed=42):
    # 受控对照：同一批 x，仅长度维取 +2 / -2，返回平均打分差
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, DIM, generator=g)
    x_long,  x_short = x.clone(), x.clone()
    x_long[:, LEN], x_short[:, LEN] = 2.0, -2.0
    with torch.no_grad():
        return (score_fn(x_long) - score_fn(x_short)).mean().item()

gap_gold   = length_gap(gold_r)
gap_clean  = length_gap(lambda x: rm_score(rm, x))
gap_biased = length_gap(lambda x: rm_score(rm_biased, x))

print(f"Δ_len  gold reward : {gap_gold:+.3f}   （偏长被扣分）")
print(f"Δ_len  干净标注 RM  : {gap_clean:+.3f}   （方向对，但幅度失真：len=±2 已是训练分布边缘，外推不可靠）")
print(f"Δ_len  带偏标注 RM  : {gap_biased:+.3f}   ← 系统性给『长』打高分")

plt.figure(figsize=(5.5, 3))
plt.bar(["gold", "RM (clean)", "RM (length-biased)"], [gap_gold, gap_clean, gap_biased],
        color=["#4caf50", "#90a4ae", "#e53935"])
plt.axhline(0, c="k", lw=0.8); plt.ylabel(r"$\Delta_{len}$（长 − 短 打分差）")
plt.title("RM 学到的是标注者，不是真奖励"); plt.tight_layout(); plt.show()

## Part 4 · Overoptimization 实验（核心）：复现 [Gao 2022] 的分叉曲线

**Best-of-n（BoN）** 是最简单的"对 RM 优化"：对每个 prompt 采 $n$ 个候选，取 **proxy RM 打分最高**的那个。$n$ 越大，优化压力越大。我们用上一节的**带偏 RM** 当 proxy（现实中的 RM 正是从有偏标注学来的），gold reward 当上帝视角：

- **proxy 曲线**：被选中候选的 RM 分数 —— 数学上必然随 $n$ 单调不降（前缀最大值）；
- **gold 曲线**：被选中候选的真实分数 —— 先升（proxy 与 gold 相关，挑高分确实变好）后降（$n$ 大时 argmax 落入 proxy 的盲区：长、且 OOD）。

优化压力的统一度量是 KL。BoN 的 KL 有解析式（与底层分布无关）：

$$ \mathrm{KL}_{\text{BoN}}(n) = \log n - \frac{n-1}{n} $$

[Gao 2022] 以 $d=\sqrt{\mathrm{KL}}$ 为横轴给出经验律 $R_{\text{BoN}}(d) = d(\alpha - \beta d)$ —— 先升后降的抛物线。我们将把两个分数都做 $n{=}1$ 基准归零（与论文同样的画法），先以 $n$ 为横轴、再以 $\sqrt{\mathrm{KL}}$ 为横轴各画一张。

> 提示：本实验全程固定随机种子。如果你换一个 RM 初始化种子重训，会发现分叉出现的位置和深度明显变化——**overoptimization 的程度取决于这个 RM 恰好在哪里外推出错**。这种 RM 间的分歧正是"研究前沿"一节里 RM ensemble 方法想利用的信号。

In [ ]:
N_TRIALS, MAX_N = 800, 256
gen = torch.Generator().manual_seed(7)
cand = torch.randn(N_TRIALS, MAX_N, DIM, generator=gen)   # 每行 = 一个 prompt 的 256 个候选
with torch.no_grad():
    proxy_scores = rm_score(rm_biased, cand)               # (trials, MAX_N)
    gold_scores  = gold_r(cand)

ns = [1, 2, 4, 8, 16, 32, 64, 128, 256]

def bon_curves(proxy_scores, gold_scores, ns):
    proxy_curve, gold_curve = [], []
    for n in ns:
        idx = proxy_scores[:, :n].argmax(dim=1)            # 每个 prompt 的 best-of-n
        rows = torch.arange(len(idx))
        proxy_curve.append(proxy_scores[rows, idx].mean().item())
        gold_curve.append(gold_scores[rows, idx].mean().item())
    return np.array(proxy_curve), np.array(gold_curve)

proxy_curve, gold_curve = bon_curves(proxy_scores, gold_scores, ns)
proxy0, gold0 = proxy_curve - proxy_curve[0], gold_curve - gold_curve[0]   # n=1 归零
kl  = np.array([math.log(n) - (n - 1) / n for n in ns])
d   = np.sqrt(kl)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(ns, proxy0, "o-", label="proxy RM 分数")
axes[0].plot(ns, gold0,  "s-", label="gold reward")
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("n (best-of-n)")
axes[1].plot(d, proxy0, "o-", label="proxy RM 分数")
axes[1].plot(d, gold0,  "s-", label="gold reward")
axes[1].set_xlabel(r"$\sqrt{KL_{BoN}}$,  $KL=\log n-(n-1)/n$")
for ax in axes:
    ax.axhline(0, c="gray", lw=0.6); ax.set_ylabel("相对 n=1 的提升"); ax.legend()
axes[0].set_title("Goodhart 分叉：proxy 单调升，gold 先升后降")
axes[1].set_title("以 KL 为优化压力度量（Gao 2022 画法）")
plt.tight_layout(); plt.show()

best_n = ns[int(np.argmax(gold0))]
print(f"gold reward 峰值出现在 n = {best_n}；之后继续优化 proxy 只会伤害真实质量（reward hacking 区）")
# 顺手观察 hacking 的"作案手法"：被选中候选的平均长度随 n 增大而被推高
rows = torch.arange(N_TRIALS)
for n in [1, 8, 64, 256]:
    idx = proxy_scores[:, :n].argmax(dim=1)
    print(f"  n={n:3d}  被选候选的平均长度特征 = {cand[rows, idx, LEN].mean():+.2f}")

## ✏️ 练习 1：实现数值稳定的 `bt_loss`

实现 Bradley-Terry 训练损失

$$ \mathcal{L} = -\frac{1}{N}\sum_i \log\sigma\big(r_c^{(i)} - r_r^{(i)}\big) $$

**要求**：
- 输入 `r_chosen, r_rejected`：同形状的分数 tensor；返回标量（mean reduction）；
- **数值稳定**：差值为 $-1000$ 时必须返回有限值（提示：用 `F.logsigmoid`，不要 `torch.log(torch.sigmoid(...))`）。

**自测会检查**：分差为 0 时 loss $=\log 2$；与 `logsigmoid` 的恒等式 $e^{-\mathcal{L}(a,b)} + e^{-\mathcal{L}(b,a)} = 1$（标量情形，即 $\sigma(d)+\sigma(-d)=1$ 的对称性）；极端分差不产生 inf/nan。

In [ ]:
def bt_loss(r_chosen, r_rejected):
    # TODO: 返回 -E[log sigmoid(r_chosen - r_rejected)]（标量，mean reduction）
    #       数值稳定：使用 F.logsigmoid
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
t = torch.tensor
# 分差为 0 → log 2
assert abs(bt_loss(t([1.0, -2.0]), t([1.0, -2.0])).item() - math.log(2)) < 1e-9
# 已知数值：d=1 → -logsigmoid(1)
assert abs(bt_loss(t([1.0]), t([0.0])).item() - 0.3132616875182228) < 1e-9
# 对称性：exp(-L(a,b)) + exp(-L(b,a)) = sigma(d) + sigma(-d) = 1（标量情形）
a, b = t([0.7]), t([-0.4])
assert abs(math.exp(-bt_loss(a, b).item()) + math.exp(-bt_loss(b, a).item()) - 1.0) < 1e-9
# mean reduction：与逐元素平均一致
assert abs(bt_loss(t([2.0, 0.0]), t([0.0, 2.0])).item()
           - (bt_loss(t([2.0]), t([0.0])).item() + bt_loss(t([0.0]), t([2.0])).item()) / 2) < 1e-9
# 数值稳定性：极端分差不爆 inf/nan
extreme = bt_loss(t([-1000.0]), t([0.0]))
assert torch.isfinite(extreme) and abs(extreme.item() - 1000.0) < 1e-6
assert torch.isfinite(bt_loss(t([1000.0]), t([0.0])))
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `rm_accuracy`

实现 RM 的偏好准确率评测（RewardBench 的核心度量）：

```
rm_accuracy(score_fn, pairs) -> float
```

- `score_fn`: 接受 `(N, DIM)` tensor、返回 `(N,)` 分数的可调用对象；
- `pairs`: 元组 `(X_chosen, X_rejected)`，两个 `(N, DIM)` tensor；
- 返回 `score_fn(chosen) > score_fn(rejected)` 的比例（float，严格大于；评测时记得 `torch.no_grad()`）。

**自测会检查**：在**完美 RM**（直接用 `gold_r`，标签也由 gold 无噪声生成）上 $\approx 1.0$；在与 gold 无关的**随机 RM** 上 $\approx 0.5$。

In [ ]:
def rm_accuracy(score_fn, pairs):
    # TODO: pairs = (X_chosen, X_rejected)
    #       返回 score_fn 给 chosen 打分严格高于 rejected 的比例（float）
    #       记得包在 torch.no_grad() 里
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
g = torch.Generator().manual_seed(123)
xa, xb = torch.randn(2000, DIM, generator=g), torch.randn(2000, DIM, generator=g)
# 无噪声标签：gold 高者为 chosen
gold_a_wins = gold_r(xa) > gold_r(xb)
Xc = torch.where(gold_a_wins.unsqueeze(-1), xa, xb)
Xr = torch.where(gold_a_wins.unsqueeze(-1), xb, xa)

acc_perfect = rm_accuracy(gold_r, (Xc, Xr))
random_rm = lambda x: torch.sin(13.7 * x[..., 5])   # 只看无关特征 x5 → 与 gold 独立
acc_random = rm_accuracy(random_rm, (Xc, Xr))

assert isinstance(acc_perfect, float) and isinstance(acc_random, float)
assert acc_perfect > 0.999, f"完美 RM 应 ≈1.0，得到 {acc_perfect}"
assert 0.45 < acc_random < 0.55, f"随机 RM 应 ≈0.5，得到 {acc_random}"
# 顺手测我们训出的 RM（应介于随机与完美之间）
acc_trained = rm_accuracy(lambda x: rm_score(rm, x), (Xc, Xr))
assert 0.6 < acc_trained < 1.0
print(f"perfect={acc_perfect:.3f}  random={acc_random:.3f}  trained={acc_trained:.3f}")
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `best_of_n_gap`

把 Part 4 的 BoN 评测封装成通用函数：

```
best_of_n_gap(rm_scores, gold_scores, ns) -> (proxy_curve, gold_curve, gaps)
```

- `rm_scores, gold_scores`: `(trials, max_n)` tensor，同一批候选在 proxy 与 gold 下的分数；
- 对每个 `n in ns`：在**前 n 列**里取 `rm_scores` 的 argmax，分别取出对应的 proxy 分数与 gold 分数，对 trials 求均值；
- 返回三个长度为 `len(ns)` 的 `np.ndarray`：proxy 均值曲线、gold 均值曲线、`gaps = proxy_curve - gold_curve`。

**自测会检查**：`proxy_curve` 随 $n$ **单调不降**（前缀最大值的性质）；`rm_scores == gold_scores` 时 `gaps ≈ 0` 且 gold 也单调不降；`n=1` 时两条曲线分别等于第一列均值。

In [ ]:
def best_of_n_gap(rm_scores, gold_scores, ns):
    # TODO: 对每个 n：idx = rm_scores[:, :n].argmax(dim=1)
    #       proxy_curve[k] = 被选中候选的 rm 分数均值
    #       gold_curve[k]  = 同一批被选中候选的 gold 分数均值
    #       返回 (proxy_curve, gold_curve, proxy_curve - gold_curve)，均为 np.ndarray
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
p_c, g_c, gaps = best_of_n_gap(proxy_scores, gold_scores, ns)
assert all(isinstance(a, np.ndarray) and len(a) == len(ns) for a in (p_c, g_c, gaps))
# proxy 随 n 单调不降（前缀最大值），允许 1e-12 浮点容差
assert np.all(np.diff(p_c) >= -1e-12), "proxy 曲线必须随 n 单调不降"
# gap 定义一致
assert np.allclose(gaps, p_c - g_c)
# n=1：两条曲线都等于第一列均值
assert abs(p_c[0] - proxy_scores[:, 0].mean().item()) < 1e-9
assert abs(g_c[0] - gold_scores[:, 0].mean().item()) < 1e-9
# proxy == gold 时无分叉：gap≈0，且 gold 同样单调不降
p_eq, g_eq, gap_eq = best_of_n_gap(gold_scores, gold_scores, ns)
assert np.allclose(gap_eq, 0.0) and np.all(np.diff(g_eq) >= -1e-12)
# 与 Part 4 的实现一致
assert np.allclose(p_c, proxy_curve) and np.allclose(g_c, gold_curve)
print("✅ 练习 3 通过")

## 📖 参考答案

In [ ]:
# 先自己做，再对照 —— 练习 1 参考答案
def bt_loss(r_chosen, r_rejected):
    return -F.logsigmoid(r_chosen - r_rejected).mean()

In [ ]:
# 先自己做，再对照 —— 练习 2 参考答案
def rm_accuracy(score_fn, pairs):
    Xc, Xr = pairs
    with torch.no_grad():
        return (score_fn(Xc) > score_fn(Xr)).double().mean().item()

In [ ]:
# 先自己做，再对照 —— 练习 3 参考答案
def best_of_n_gap(rm_scores, gold_scores, ns):
    proxy_curve, gold_curve = [], []
    rows = torch.arange(rm_scores.shape[0])
    for n in ns:
        idx = rm_scores[:, :n].argmax(dim=1)
        proxy_curve.append(rm_scores[rows, idx].mean().item())
        gold_curve.append(gold_scores[rows, idx].mean().item())
    proxy_curve, gold_curve = np.array(proxy_curve), np.array(gold_curve)
    return proxy_curve, gold_curve, proxy_curve - gold_curve

## 小结

| 实验 | 现象 | 对应真实世界 |
|---|---|---|
| BT loss 训练 | held-out 准确率逼近但到不了天花板 | RM 准确率 65–75% 是常态，受标注噪声限制 |
| 长度偏好注入 | RM 的 $\Delta_{\text{len}}$ 系统性为正 | RLHF 后模型变啰嗦；RewardBench Chat-Hard 专测此项 |
| BoN overoptimization | proxy 单调升、gold 先升后降 | [Gao 2022] 分叉曲线；Goodhart 定律 |
| KL 横轴 | $\mathrm{KL}_{\text{BoN}} = \log n - \frac{n-1}{n}$ | 优化压力的统一度量，PPO 的 KL 惩罚由此而来 |

核心带走一句话：**RM 是标注者的极大似然代理，对它优化得越狠，你拿到的越是它的盲区而非人类的偏好**。控制优化压力（KL 惩罚、早停、BoN 的 n）是 RLHF 的第一道安全阀。

**下一站 · 模块 03（RLHF 与 PPO）**：把这里的 BoN 换成真正的 RL —— policy gradient 直接最大化 $r_\phi$，KL 惩罚从"画图的横轴"变成"目标函数里的一项"，你将看到同样的 overoptimization 以更隐蔽的方式出现。

---
## 🎯 真实数据胶囊题：真实红酒质量上的 Bradley-Terry 偏好建模

Bradley-Terry 从两两胜负里恢复每个对象的“能力分”。把 UCI 红酒按质量分级当“选手”，用真实质量差生成两两偏好（高质量更可能赢），用 BT 拟合能力分，验证它单调反映真实质量。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.post_training_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=200):
    p=_fetch("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    rows=[json.loads(l) for l in open(p).read().splitlines()[:n]]
    return rows
def winequality():
    import pandas as pd
    p=_fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
    return pd.read_csv(p, sep=";")

df = winequality()
levels = sorted(df["quality"].unique())   # 真实质量等级 3..8
true_q = {q:i for i,q in enumerate(levels)}
print("真实红酒质量等级:", levels)
# 生成两两比较：质量高的以更高概率赢（用真实等级差）
rng=np.random.default_rng(0)
def sample_battles(n=4000):
    out=[]
    for _ in range(n):
        a,b = rng.choice(levels,2,replace=False)
        pa = 1/(1+np.exp(-(a-b)))           # 真实质量差 -> 胜率
        win_a = rng.random() < pa
        out.append((true_q[a], true_q[b], 1 if win_a else 0))
    return out
battles=sample_battles()
print(f"{len(battles)} 场两两比较，{len(levels)} 个质量等级")

**练习**：实现 `fit_bradley_terry(battles, n_items, steps, lr)`：梯度上升拟合每个 item 的能力分 `s`，胜率模型 `P(i>j)=sigmoid(s_i-s_j)`。返回能力向量（中心化）。

In [ ]:
def fit_bradley_terry(battles, n_items, steps=2000, lr=0.05):
    # TODO: s=zeros(n_items); 对每个 (i,j,win_i): p=sigmoid(s_i-s_j); grad: (win_i-p) 加到 s_i 减到 s_j
    raise NotImplementedError


In [ ]:
# 自测：恢复的能力分应随真实质量单调递增
s = fit_bradley_terry(battles, len(levels))
assert len(s)==len(levels)
# 能力分排序应与质量等级一致（Spearman 单调）
order = np.argsort(s)
assert list(order)==list(range(len(levels))), f"BT 能力应随质量单调，得到 {s.round(2)}"
print(f"BT 恢复的能力分(按质量排) = {s.round(2)} ✓ 单调递增")


### 📖 参考答案

In [ ]:
def fit_bradley_terry(battles, n_items, steps=2000, lr=0.05):
    s=np.zeros(n_items)
    for _ in range(steps):
        g=np.zeros(n_items)
        for i,j,wi in battles:
            p=1/(1+np.exp(-(s[i]-s[j]))); g[i]+=(wi-p); g[j]-=(wi-p)
        s+=lr*g/len(battles); s-=s.mean()
    return s
print("✓ reward model 的数学内核就是 Bradley-Terry")